In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# 1. Cargar el dataset original temporalmente (Instrucción de Jorge)
ruta_archivo = "../../data/raw/20260817_Titulados_Ed_Superior_2025_WEB.csv"
df = pd.read_csv(ruta_archivo, sep=';', encoding='utf-8')
df.columns = df.columns.str.lower()

# Filtrar a la población de estudio (Pregrado Universitario)
filtro = (df['nivel_global'] == 'Pregrado') & (df['tipo_inst_1'] == 'Universidades')
df = df[filtro].copy()

# 2. Cálculo del Atraso real ("como en la S2") 
# Usamos el año de titulación (cat_periodo) menos el año de ingreso, multiplicado por 2 semestres, 
# menos la duración formal de la carrera (dur_estudio_carr)
if 'cat_periodo' in df.columns and 'anio_ing_carr_ori' in df.columns and 'dur_estudio_carr' in df.columns:
    df['semestres_reales'] = (df['cat_periodo'] - df['anio_ing_carr_ori']) * 2
    df['atraso_semestres'] = df['semestres_reales'] - df['dur_estudio_carr']
else:
    print("Advertencia: Faltan columnas para calcular el atraso. Revisa los nombres de las variables de la S2.")

# 3. Aplicar dropna temporalmente (Instrucción de Jorge)
columnas_necesarias = ['gen_alu', 'area_conocimiento', 'jornada', 'anio_ing_carr_ori', 'fec_nac_alu', 'atraso_semestres']
df = df.dropna(subset=[col for col in columnas_necesarias if col in df.columns])

# 4. Ajustes requeridos para el modelo (Género y Edad de Ingreso)
if pd.api.types.is_numeric_dtype(df['gen_alu']):
    df['gen_alu'] = df['gen_alu'].map({1: 'Hombre', 2: 'Mujer'})

df['edad_ingreso'] = df['anio_ing_carr_ori'] - pd.to_datetime(df['fec_nac_alu'], format='%Y%m%d', errors='coerce').dt.year
df = df.dropna(subset=['edad_ingreso']) # Limpiar posibles nulos generados en la edad

# 5. Definir Target como "atraso alto" (sobre Q3)
Q3_atraso = df['atraso_semestres'].quantile(0.75)
df['target_atraso_alto'] = (df['atraso_semestres'] > Q3_atraso).astype(int)

print(f"Umbral Q3 para atraso alto: {Q3_atraso} semestres")
print(f"Distribución del Target (% en clase 1): {df['target_atraso_alto'].mean() * 100:.2f}%")

# 6. Preparar variables para el modelo
variables_base = ['gen_alu', 'area_conocimiento', 'jornada', 'edad_ingreso']
X_bruto = df[variables_base].copy()

X_procesado = pd.get_dummies(X_bruto, drop_first=True, dtype=int)
X_final = sm.add_constant(X_procesado)
y = df['target_atraso_alto']

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
print("\nDatos preparados exitosamente con 'dropna' temporal. Tamaño:", X_train.shape)

In [ ]:
# PUNTO 6: Modelo 1 (Variables S1/S2: sexo, área, jornada)
# Excluimos la edad_ingreso para el modelo base según la instrucción
cols_modelo_1 = [col for col in X_train.columns if 'edad' not in col]
modelo_base = sm.Logit(y_train, X_train[cols_modelo_1]).fit(disp=False)

# PUNTO 6: Modelo 2 (Stepwise Selection)
def forward_selection(data, target, significance_level=0.05):
    initial_features = data.columns.tolist()
    best_features = []
    while (len(initial_features) > 0):
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features, dtype=float)
        for new_column in remaining_features:
            model = sm.Logit(target, data[best_features + [new_column]]).fit(disp=False)
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

cols_stepwise = forward_selection(X_train, y_train)
if 'const' not in cols_stepwise: 
    cols_stepwise.insert(0, 'const')

modelo_stepwise = sm.Logit(y_train, X_train[cols_stepwise]).fit(disp=False)

# PUNTO 6: Comparación de AIC/BIC
print("--- COMPARACIÓN DE MODELOS ---")
print(f"Modelo Base (S1/S2): AIC = {modelo_base.aic:.2f} | BIC = {modelo_base.bic:.2f}")
print(f"Modelo Stepwise:     AIC = {modelo_stepwise.aic:.2f} | BIC = {modelo_stepwise.bic:.2f}")

# Seleccionamos el mejor modelo para continuar
mejor_modelo = modelo_stepwise if modelo_stepwise.aic < modelo_base.aic else modelo_base
cols_mejor = cols_stepwise if modelo_stepwise.aic < modelo_base.aic else cols_modelo_1
print(f"\nModelo seleccionado automáticamente: {'Stepwise' if modelo_stepwise.aic < modelo_base.aic else 'Base'}")

In [ ]:
# PUNTO 7: Bootstrap de Odds Ratios remuestreando por institución
n_iterations = 500

# IMPORTANTE: Revisa cómo se llama la columna de institución en tu dataset real (ej: 'nomb_inst', 'cod_inst')
col_institucion = 'nomb_inst' if 'nomb_inst' in df.columns else 'codigo_institucion' 

df_train = X_train[cols_mejor].copy()
df_train['target'] = y_train
# Rescatamos la institución desde el dataframe original usando los índices del train
df_train[col_institucion] = df.loc[X_train.index, col_institucion]

instituciones_unicas = df_train[col_institucion].dropna().unique()
coef_bootstrap = {col: [] for col in cols_mejor}

print(f"Ejecutando {n_iterations} iteraciones de Bootstrap agrupado por {col_institucion}...\n")

for i in range(n_iterations):
    # Remuestreo con reemplazo a nivel de institución
    inst_muestra = np.random.choice(instituciones_unicas, size=len(instituciones_unicas), replace=True)
    df_boot = pd.concat([df_train[df_train[col_institucion] == inst] for inst in inst_muestra])
    
    try:
        model_boot = sm.Logit(df_boot['target'], df_boot[cols_mejor]).fit(disp=False)
        for col in cols_mejor:
            coef_bootstrap[col].append(model_boot.params[col])
    except:
        continue # Ignora la iteración si el remuestreo genera separación perfecta de clases

# Cálculo de exp(coef) y sus IC al 95%
resultados_or = []
for col in cols_mejor:
    if len(coef_bootstrap[col]) > 0:
        coefs = np.array(coef_bootstrap[col])
        or_mean = np.exp(np.mean(coefs))
        or_lower = np.exp(np.percentile(coefs, 2.5))
        or_upper = np.exp(np.percentile(coefs, 97.5))
        resultados_or.append({'Variable': col, 'Odds Ratio': round(or_mean, 3), 
                              'IC 2.5%': round(or_lower, 3), 'IC 97.5%': round(or_upper, 3)})

print("--- ODDS RATIOS E INTERVALOS DE CONFIANZA (BOOTSTRAP 95%) ---")
print(pd.DataFrame(resultados_or).set_index('Variable'))

In [ ]:
# PUNTO 8: Evaluación con Curva ROC (Youden), Sensibilidad y Especificidad

# Predicciones probabilísticas en el set de prueba
y_pred_prob = mejor_modelo.predict(X_test[cols_mejor])

# Cálculo de ROC y métricas
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Índice de Youden (Sensibilidad + Especificidad - 1 = TPR - FPR)
youden_index = tpr - fpr
best_threshold_idx = np.argmax(youden_index)
best_threshold = thresholds[best_threshold_idx]

# Clasificación final con el umbral óptimo
y_pred_optimo = (y_pred_prob >= best_threshold).astype(int)

# Extraer valores de la matriz de confusión para calcular métricas manualmente
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_optimo).ravel()
sensibilidad = tp / (tp + fn)
especificidad = tn / (tn + fp)

print("--- EVALUACIÓN FINAL CON UMBRAL OPTIMIZADO (YOUDEN) ---")
print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Umbral Óptimo de Youden: {best_threshold:.4f}")
print(f"Sensibilidad (Tasa de Verdaderos Positivos): {sensibilidad:.4f}")
print(f"Especificidad (Tasa de Verdaderos Negativos): {especificidad:.4f}\n")

print("Matriz de Confusión (Umbral Youden):")
print(pd.DataFrame([[tn, fp], [fn, tp]], 
                   columns=['Predice No Atraso', 'Predice Atraso Alto'], 
                   index=['Real No Atraso', 'Real Atraso Alto']))

print("\nReporte de Clasificación (Umbral Youden):")
print(classification_report(y_test, y_pred_optimo))